# AI/ML-Based Prediction of 5G Downlink Throughput Using Channel and Context Metrics
**QM640 Data Analytics Capstone, Lokesh Rajoria, Walsh College**

This notebook reproduces the analysis in the interim report, in the same order: data acquisition, cleaning, exploratory analysis, RQ1 correlations, model comparison (RQ2-RQ4) and feature importance.

**How to run:** *Runtime > Run all*. Everything runs in Colab; the public dataset is downloaded from the authors' GitHub repository (Raca et al., 2020).

**Run time:** about 10-15 minutes in quick mode (default). Quick mode searches only the model settings that the full grid selected most often; set `QUICK = False` below to run the full nested grid (about 45-60 minutes on a standard Colab CPU). Numbers in quick mode can differ slightly from the report.

In [ ]:
QUICK = True                 # False = full nested grid search, as in the report
RUN_SENSITIVITY = False      # True = also hold out whole recording dates and the latest dates (slow)
RUN_IMPORTANCE = True        # permutation feature importance (about 3-5 minutes)

import os, sys, logging
os.makedirs('/content/capstone', exist_ok=True)
%cd /content/capstone
sys.path.insert(0, os.getcwd())   # make the tput package importable
os.environ['TPUT_QUICK'] = '1' if QUICK else '0'
logging.getLogger('matplotlib.font_manager').setLevel(logging.ERROR)
import pandas as pd, sklearn
pd.set_option('display.width', 200); pd.set_option('display.max_columns', 30)
print('Python', sys.version.split()[0], '| pandas', pd.__version__, '| scikit-learn', sklearn.__version__)

## 1. Analysis package
The next cells write the project's Python modules into the runtime (`tput/` package and the run scripts). They are identical to the files in the submitted analysis folder. You can collapse this section.

In [ ]:
os.makedirs('tput', exist_ok=True)

In [ ]:
%%writefile tput/__init__.py
"""5G downlink throughput prediction: data preparation, evaluation splits and baselines."""


In [ ]:
%%writefile tput/config.py
"""Paths and fixed study settings (synopsis sections 4-6). Change these only with a documented reason."""
from pathlib import Path

ROOT = Path(__file__).resolve().parents[1]
RAW_ZIP = ROOT / "data" / "raw" / "5G-production-dataset.zip"
RAW_DIR = ROOT / "data" / "raw" / "5G-production-dataset"
INTERIM = ROOT / "data" / "interim"
PROCESSED = ROOT / "data" / "processed"
TABLES = ROOT / "outputs" / "tables"
FIGURES = ROOT / "outputs" / "figures"

# Expected input (verified in the preliminary audit). The pipeline stops if the archive differs.
RAW_ZIP_SHA256 = "abc729d696b5e0ba34a9c6ed35a851f2868d2b5071be5efb8a49691c23bb1a9b"
EXPECTED_TRACES = 83
EXPECTED_ROWS = 188_711

SEED = 640
HORIZON_S = 5                 # predict throughput exactly 5 s after the origin
LAGS_S = (0, 1, 2, 5)         # throughput_lag_k = throughput exactly k s before the origin
WINDOW_S = 10                 # throughput_mean_10s / throughput_sd_10s over seconds t-9 .. t
# Eligibility rule. The target and lag_0 are always exact. ~10 % of seconds are missing from the logs,
# so demanding all 10 window seconds and every lag keeps only 19 % of origins (4-31 % by group).
# Default: window needs >= 7 observed seconds; missing lag_1/2/5 are imputed with indicators.
# Strict rule (sensitivity analysis): WINDOW_MIN_OBS = 10 and REQUIRE_ALL_LAGS = True.
WINDOW_MIN_OBS = 7
REQUIRE_ALL_LAGS = False
OUTER_FOLDS = 5
INNER_FOLDS = 3
MIN_PAIRS_RQ1 = 30

# 3GPP TS 38.133 / 36.133 RSRP reporting floor; the logger writes -200 as a "no value" marker.
RSRP_MIN_VALID = -156

RADIO = ["RSRP", "RSRQ", "SNR", "CQI"]
HISTORY = [f"throughput_lag_{k}" for k in LAGS_S] + ["throughput_mean_10s", "throughput_sd_10s"]
CONTEXT_NUM = ["Speed"]
CONTEXT_CAT = ["NetworkMode", "application_type"]
BASE_FEATURES = RADIO + HISTORY
CONTEXT_FEATURES = BASE_FEATURES + CONTEXT_NUM + CONTEXT_CAT
TARGET = "target_throughput_5s"


In [ ]:
%%writefile tput/manifest.py
"""Step 1: verify the archive and build the trace manifest (one row per CSV trace)."""
import hashlib

import pandas as pd

from . import config as C


def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1 << 20), b""):
            h.update(block)
    return h.hexdigest()


def build_manifest():
    digest = sha256(C.RAW_ZIP)
    if digest != C.RAW_ZIP_SHA256:
        raise RuntimeError(f"Archive checksum {digest} does not match the recorded input version")

    rows = []
    for path in sorted(C.RAW_DIR.rglob("*.csv")):
        rel = path.relative_to(C.RAW_DIR)
        parts = rel.parts                       # App / Static|Driving / [content /] file.csv
        ts = pd.read_csv(path, usecols=["Timestamp"]).Timestamp
        t = pd.to_datetime(ts, format="%Y.%m.%d_%H.%M.%S")
        rows.append({
            "trace_id": rel.as_posix(),
            "application_type": parts[0].replace("_", " "),
            "mobility_condition": parts[1],
            "content": parts[2] if len(parts) == 4 else "",
            "recording_date": t.min().date().isoformat(),
            "start_time": t.min(),
            "n_raw_rows": len(ts),
            "duration_s": int((t.max() - t.min()).total_seconds()),
        })
    m = pd.DataFrame(rows)
    if len(m) != C.EXPECTED_TRACES or m.n_raw_rows.sum() != C.EXPECTED_ROWS:
        raise RuntimeError(f"Expected {C.EXPECTED_TRACES} traces / {C.EXPECTED_ROWS} rows, "
                           f"found {len(m)} / {m.n_raw_rows.sum()}")
    m.attrs["sha256"] = digest
    return m


In [ ]:
%%writefile tput/clean.py
"""Step 2: parse, mark invalid values, and consolidate repeated timestamps to one row per second.

Rules (each one is counted in the exclusion log):
  - literal "-" -> missing
  - RSRP below the 3GPP reporting floor (the logger's -200 marker) -> missing
  - rows sharing a timestamp within a trace -> one row: numeric fields averaged,
    categorical fields take the last value, n_merged records how many rows were combined
  - zero throughput is kept: it can mean the application was idle, not that the link failed
"""
import pandas as pd

from . import config as C

NUMERIC = ["Longitude", "Latitude", "Speed", "RSRP", "RSRQ", "SNR", "CQI", "RSSI", "DL_bitrate", "UL_bitrate",
           "PINGAVG", "PINGMIN", "PINGMAX", "PINGSTDEV", "PINGLOSS", "NRxRSRP", "NRxRSRQ"]
CATEGORICAL = ["Operatorname", "CellID", "NetworkMode", "State", "CELLHEX", "NODEHEX", "LACHEX", "RAWCELLID"]


def load_trace(path):
    d = pd.read_csv(path, na_values=["-"], keep_default_na=False, dtype=str)
    d["t"] = pd.to_datetime(d.pop("Timestamp"), format="%Y.%m.%d_%H.%M.%S")
    for c in NUMERIC:
        d[c] = pd.to_numeric(d[c], errors="raise")
    return d


def clean_all(manifest):
    log, frames = [], []
    for tr in manifest.itertuples():
        d = load_trace(C.RAW_DIR / tr.trace_id)
        bad_rsrp = d.RSRP < C.RSRP_MIN_VALID
        d.loc[bad_rsrp, "RSRP"] = float("nan")

        agg = {c: "mean" for c in NUMERIC} | {c: "last" for c in CATEGORICAL}
        g = d.groupby("t", sort=True)
        s = g.agg(agg)
        s["n_merged"] = g.size()
        s = s.reset_index()
        s.insert(0, "trace_id", tr.trace_id)
        s["throughput_mbps"] = s.DL_bitrate / 1000.0
        frames.append(s)

        steps = s.t.diff().dt.total_seconds()
        log.append({"trace_id": tr.trace_id, "raw_rows": len(d), "rsrp_invalid": int(bad_rsrp.sum()),
                    "rows_merged_away": len(d) - len(s), "seconds_kept": len(s),
                    "gaps_over_1s": int((steps > 1).sum()), "missing_seconds": int((steps - 1).clip(lower=0).sum())})
    clean = pd.concat(frames, ignore_index=True)
    clean = clean.merge(manifest[["trace_id", "application_type", "mobility_condition", "recording_date"]],
                        on="trace_id", how="left", validate="many_to_one")
    return clean, pd.DataFrame(log)


In [ ]:
%%writefile tput/features.py
"""Step 3: exact 5 s-ahead target and history features, using only data at or before the origin.

Every lookup is by exact timestamp inside the same trace, so a gap in the log gives a missing value
instead of silently pairing readings that are further apart than intended. Which missing values make an
origin ineligible is set by WINDOW_MIN_OBS / REQUIRE_ALL_LAGS in config.
"""
import warnings

import numpy as np
import pandas as pd

from . import config as C


def _at(series, times):
    """Value of a time-indexed series at each requested timestamp (NaN where that second is absent)."""
    return series.reindex(times).to_numpy()


def build_features(clean):
    out = []
    for _, d in clean.groupby("trace_id", sort=False):
        d = d.sort_values("t").reset_index(drop=True)
        y = d.set_index("t").throughput_mbps
        t = d.t
        f = d[["trace_id", "t", "application_type", "mobility_condition", "recording_date",
               "NetworkMode", "Speed", *C.RADIO]].copy()

        for k in C.LAGS_S:
            f[f"throughput_lag_{k}"] = _at(y, t - pd.Timedelta(seconds=k))
        win = np.column_stack([_at(y, t - pd.Timedelta(seconds=k)) for k in range(C.WINDOW_S)])
        n_obs = (~np.isnan(win)).sum(axis=1)
        enough = n_obs >= C.WINDOW_MIN_OBS
        with np.errstate(invalid="ignore"), warnings.catch_warnings():
            warnings.simplefilter("ignore", RuntimeWarning)          # all-NaN rows are masked below
            f["throughput_mean_10s"] = np.where(enough, np.nanmean(win, axis=1), np.nan)
            f["throughput_sd_10s"] = np.where(enough, np.nanstd(win, axis=1), np.nan)  # divisor = seconds observed
        f["window_n_obs"] = n_obs
        f[C.TARGET] = _at(y, t + pd.Timedelta(seconds=C.HORIZON_S))
        out.append(f)
    feats = pd.concat(out, ignore_index=True)

    required = [C.TARGET, "throughput_lag_0", "throughput_mean_10s", "throughput_sd_10s"]
    if C.REQUIRE_ALL_LAGS:
        required += [f"throughput_lag_{k}" for k in C.LAGS_S]
    feats["eligible"] = feats[required].notna().all(axis=1)
    for c in C.RADIO + ["Speed"] + [f"throughput_lag_{k}" for k in C.LAGS_S if k]:
        feats[f"{c}_missing"] = feats[c].isna().astype(int)
    return feats


def coverage(feats):
    """How many origins survive the exact-timing rules, by group (synopsis step 3 check)."""
    g = feats.groupby(["application_type", "mobility_condition"])
    return pd.DataFrame({
        "traces": g.trace_id.nunique(),
        "origins": g.size(),
        "with_target": g[C.TARGET].apply(lambda s: s.notna().sum()),
        "eligible": g.eligible.sum(),
    }).assign(eligible_pct=lambda x: (100 * x.eligible / x.origins).round(1)).reset_index()


In [ ]:
%%writefile tput/splits.py
"""Step 4: trace-level evaluation partitions, fixed before any model is fitted.

  outer_fold  - 5 folds of whole traces, stratified by application x mobility (primary evaluation)
  date_fold   - whole recording dates held out together (date-dependence sensitivity)
  chrono_test - traces from the latest dates held out (chronological sensitivity)
Inner 3-fold model selection uses inner_folds() on the outer training traces only.
"""
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupKFold, StratifiedGroupKFold

from . import config as C


def make_splits(manifest, eligible_traces):
    m = manifest[manifest.trace_id.isin(eligible_traces)].reset_index(drop=True)
    strata = m.application_type + "|" + m.mobility_condition

    outer = StratifiedGroupKFold(n_splits=C.OUTER_FOLDS, shuffle=True, random_state=C.SEED)
    m["outer_fold"] = -1
    for k, (_, test) in enumerate(outer.split(m, strata, groups=m.trace_id)):
        m.loc[test, "outer_fold"] = k

    m["date_fold"] = -1
    for k, (_, test) in enumerate(GroupKFold(n_splits=C.OUTER_FOLDS).split(m, groups=m.recording_date)):
        m.loc[test, "date_fold"] = k

    dates = np.sort(m.recording_date.unique())
    cutoff = dates[int(np.ceil(0.8 * len(dates)))]          # latest ~20 % of recording dates
    m["chrono_test"] = (m.recording_date >= cutoff).astype(int)
    return m[["trace_id", "application_type", "mobility_condition", "recording_date",
              "outer_fold", "date_fold", "chrono_test"]]


def inner_folds(train_traces, strata):
    """Inner grouped folds for model selection inside one outer training partition."""
    cv = StratifiedGroupKFold(n_splits=C.INNER_FOLDS, shuffle=True, random_state=C.SEED)
    return list(cv.split(train_traces, strata, groups=train_traces))


In [ ]:
%%writefile tput/evaluate.py
"""Trace-level error metrics and the persistence baseline (synopsis 5.2 and 5.5)."""
import numpy as np
import pandas as pd

from . import config as C


def persistence(feats):
    """Predicted throughput at t + 5 s = observed throughput at t."""
    return feats["throughput_lag_0"].to_numpy()


def trace_errors(feats, pred, name):
    e = feats[["trace_id", "application_type", "mobility_condition", C.TARGET]].assign(pred=pred)
    e["abs_err"] = (e.pred - e[C.TARGET]).abs()
    e["sq_err"] = (e.pred - e[C.TARGET]) ** 2
    g = e.groupby(["trace_id", "application_type", "mobility_condition"])
    return pd.DataFrame({"n": g.size(), "mae": g.abs_err.mean(), "rmse": np.sqrt(g.sq_err.mean())}) \
        .reset_index().assign(model=name)


def summarise(per_trace, by=("mobility_condition",)):
    """Trace-weighted means (each trace counts once), the primary aggregate in the synopsis."""
    rows = [per_trace.assign(group="all")] + [per_trace.assign(group=per_trace[b]) for b in by]
    s = pd.concat(rows).groupby(["model", "group"]).agg(
        traces=("trace_id", "nunique"), origins=("n", "sum"), mean_trace_mae=("mae", "mean"),
        mean_trace_rmse=("rmse", "mean"))
    return s.round(3).reset_index()


In [ ]:
%%writefile tput/checks.py
"""Integrity checks from synopsis 6.3. Each raises AssertionError on failure."""
import numpy as np
import pandas as pd

from . import config as C


def check_no_leakage(clean, feats, n=2000, seed=C.SEED):
    """Recompute a random sample of rows straight from the cleaned data and compare."""
    rng = np.random.default_rng(seed)
    y = clean.set_index(["trace_id", "t"]).throughput_mbps
    sample = feats.iloc[rng.choice(len(feats), size=min(n, len(feats)), replace=False)]
    for r in sample.itertuples():
        def at(dt):
            return y.get((r.trace_id, r.t + pd.Timedelta(seconds=dt)), np.nan)
        assert np.allclose(getattr(r, C.TARGET), at(C.HORIZON_S), equal_nan=True), "target mismatch"
        for k in C.LAGS_S:
            assert np.allclose(getattr(r, f"throughput_lag_{k}"), at(-k), equal_nan=True), f"lag_{k} mismatch"


def check_splits(splits):
    assert splits.trace_id.is_unique, "a trace appears in more than one outer fold"
    assert set(splits.outer_fold) == set(range(C.OUTER_FOLDS)), "missing outer fold"
    for k in range(C.OUTER_FOLDS):
        test = set(splits.trace_id[splits.outer_fold == k])
        train = set(splits.trace_id[splits.outer_fold != k])
        assert not test & train, f"fold {k}: evaluation trace also in training"
    for col in ("date_fold",):
        per_date = splits.groupby("recording_date")[col].nunique()
        assert (per_date == 1).all(), f"{col}: a recording date is split across folds"


In [ ]:
%%writefile tput/stats.py
"""Uncertainty helpers. Traces (or recording dates) are the resampling units, never rows."""
import numpy as np
import pandas as pd

from . import config as C


def holm(pvals):
    """Holm step-down adjusted p-values, in the original order."""
    p = np.asarray(pvals, float)
    order = np.argsort(p)
    adj = np.empty_like(p)
    running = 0.0
    for rank, i in enumerate(order):
        running = max(running, (len(p) - rank) * p[i])
        adj[i] = min(1.0, running)
    return adj


def bootstrap_mean(values, clusters=None, n_boot=5000, level=0.95, seed=C.SEED):
    """Mean with a percentile bootstrap CI. If clusters are given (e.g. recording dates),
    whole clusters are resampled so traces sharing a date move together."""
    v = pd.Series(np.asarray(values, float))
    rng = np.random.default_rng(seed)
    if clusters is None:
        idx = rng.integers(0, len(v), size=(n_boot, len(v)))
        boots = v.to_numpy()[idx].mean(axis=1)
    else:
        groups = [g.to_numpy() for _, g in v.groupby(np.asarray(clusters))]
        k = len(groups)
        boots = np.array([np.concatenate([groups[j] for j in rng.integers(0, k, k)]).mean() for _ in range(n_boot)])
    a = (1 - level) / 2
    return {"mean": v.mean(), "ci_low": np.quantile(boots, a), "ci_high": np.quantile(boots, 1 - a),
            "share_boot_le_0": float((boots <= 0).mean()), "n": len(v)}


def bootstrap_diff(a, b, n_boot=5000, level=0.95, seed=C.SEED):
    """Difference in means (a - b) between two independent groups of traces."""
    a, b = np.asarray(a, float), np.asarray(b, float)
    rng = np.random.default_rng(seed)
    boots = (a[rng.integers(0, len(a), (n_boot, len(a)))].mean(axis=1)
             - b[rng.integers(0, len(b), (n_boot, len(b)))].mean(axis=1))
    q = (1 - level) / 2
    return {"diff": a.mean() - b.mean(), "ci_low": np.quantile(boots, q), "ci_high": np.quantile(boots, 1 - q),
            "n_a": len(a), "n_b": len(b)}


In [ ]:
%%writefile tput/rq1.py
"""RQ1: within-trace Spearman correlation between each radio measurement and contemporaneous throughput.

Eligibility per trace and measurement: >= 30 complete pairs and variation in both variables; no imputation.
Inference treats traces as the units (one-sample t-test on trace correlations, Holm across the four
measurements). The date sensitivity averages traces within each recording date first.
"""
import numpy as np
import pandas as pd
from scipy import stats

from . import config as C
from .stats import holm


def trace_correlations(clean):
    rows = []
    for tid, d in clean.groupby("trace_id"):
        meta = d.iloc[0]
        for m in C.RADIO:
            pair = d[[m, "throughput_mbps"]].dropna()
            ok = len(pair) >= C.MIN_PAIRS_RQ1 and pair[m].nunique() > 1 and pair.throughput_mbps.nunique() > 1
            r = stats.spearmanr(pair[m], pair.throughput_mbps).statistic if ok else np.nan
            rows.append({"trace_id": tid, "measurement": m, "n_pairs": len(pair), "eligible": ok, "rho": r,
                         "application_type": meta.application_type, "mobility_condition": meta.mobility_condition,
                         "recording_date": meta.recording_date})
    return pd.DataFrame(rows)


def _ttest(x):
    x = np.asarray(x, float)
    t = stats.ttest_1samp(x, 0.0)
    ci = t.confidence_interval(0.95)
    return x.mean(), ci.low, ci.high, t.pvalue, len(x)


def summarise(corr):
    out = []
    for m in C.RADIO:
        c = corr[(corr.measurement == m) & corr.eligible]
        mean, lo, hi, p, n = _ttest(c.rho)
        by_date = c.groupby("recording_date").rho.mean()
        _, dlo, dhi, dp, dn = _ttest(by_date)
        out.append({"measurement": m, "eligible_traces": n,
                    "excluded_traces": int((corr.measurement == m).sum() - n),
                    "mean_rho": mean, "ci95_low": lo, "ci95_high": hi, "p": p,
                    "share_positive": (c.rho > 0).mean(),
                    "dates": dn, "date_mean_rho": by_date.mean(), "date_ci95_low": dlo, "date_ci95_high": dhi,
                    "date_p": dp,
                    "mean_rho_static": c[c.mobility_condition == "Static"].rho.mean(),
                    "mean_rho_driving": c[c.mobility_condition == "Driving"].rho.mean()})
    s = pd.DataFrame(out)
    s["p_holm"] = holm(s.p)
    s["date_p_holm"] = holm(s.date_p)
    return s


In [ ]:
%%writefile tput/models.py
"""RQ2/RQ3 models and nested grouped cross-validation (synopsis 5.2, 5.3, 5.5).

Per outer fold:
  1. Inner grouped CV on the outer training traces tunes every candidate of every family (base features).
  2. The family with the lowest inner trace-mean MAE is the fold's selected AI/ML approach (RQ2).
  3. That same family is re-tuned on the context features with the same inner folds and grid (RQ3).
  4. Tuned models are refitted on the full outer training data and predict the held-out traces.
All preprocessing lives inside the sklearn pipeline, so it is fitted on training rows only.
"""
import os
import time

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from . import config as C
from .splits import inner_folds

FLAGS = [f"{c}_missing" for c in C.RADIO] + [f"throughput_lag_{k}_missing" for k in C.LAGS_S if k]
FEATURE_SETS = {
    "base": {"num": C.RADIO + C.HISTORY + FLAGS, "cat": []},
    "context": {"num": C.RADIO + C.HISTORY + FLAGS + C.CONTEXT_NUM, "cat": C.CONTEXT_CAT},
}

GRIDS = {
    "ridge": [{"alpha": a} for a in (0.1, 1.0, 10.0, 100.0)],
    "rf": [{"min_samples_leaf": l, "max_features": m} for l in (5, 20) for m in (0.5, 1.0)],
    "hgb": [{"loss": l, "learning_rate": r} for l in ("squared_error", "absolute_error") for r in (0.05, 0.1)],
}
if os.environ.get("TPUT_QUICK") == "1":
    # Demonstration mode (Colab notebook): search only the settings most often selected by the full grid.
    GRIDS = {"ridge": [{"alpha": 0.1}], "rf": [{"min_samples_leaf": 20, "max_features": 1.0}],
             "hgb": [{"loss": "absolute_error", "learning_rate": 0.1}]}


def make_model(family, params, fset):
    cols = FEATURE_SETS[fset]
    num = [SimpleImputer(strategy="median")] + ([StandardScaler()] if family == "ridge" else [])
    steps = [("num", make_pipeline(*num), cols["num"])]
    if cols["cat"]:
        steps.append(("cat", OneHotEncoder(handle_unknown="ignore", min_frequency=50), cols["cat"]))
    pre = ColumnTransformer(steps)
    if family == "ridge":
        est = Ridge(**params)
    elif family == "rf":
        est = RandomForestRegressor(n_estimators=200, max_samples=0.3, n_jobs=-1, random_state=C.SEED, **params)
    else:
        est = HistGradientBoostingRegressor(max_iter=300, max_leaf_nodes=31, early_stopping=False,
                                            random_state=C.SEED, **params)
    return Pipeline([("pre", pre), ("model", est)])


def trace_mean_mae(trace_ids, y, pred):
    e = pd.DataFrame({"t": np.asarray(trace_ids), "ae": np.abs(np.asarray(y) - pred)})
    return e.groupby("t").ae.mean().mean()


def _inner_scores(train, folds, family, fset, grid):
    scores = []
    for params in grid:
        s = []
        for tr_idx, va_idx in folds:
            tr, va = train.iloc[tr_idx], train.iloc[va_idx]
            m = make_model(family, params, fset).fit(tr, tr[C.TARGET])
            s.append(trace_mean_mae(va.trace_id, va[C.TARGET], m.predict(va)))
        scores.append(float(np.mean(s)))
    best = int(np.argmin(scores))
    return grid[best], scores[best], scores


def _row_folds(train, group_col):
    """Inner folds built on whole groups (traces, or dates for the date split), mapped to row indices."""
    units = train.drop_duplicates(group_col)[[group_col, "application_type", "mobility_condition"]]
    strata = (units.application_type + "|" + units.mobility_condition).to_numpy()
    unit_ids = units[group_col].to_numpy()
    out = []
    for tr_u, va_u in inner_folds(unit_ids, strata):
        tr_set, va_set = set(unit_ids[tr_u]), set(unit_ids[va_u])
        out.append((np.flatnonzero(train[group_col].isin(tr_set)), np.flatnonzero(train[group_col].isin(va_set))))
    return out


def run_nested_cv(feats, splits, split_col="outer_fold", log=print):
    data = feats[feats.eligible].merge(splits[["trace_id", split_col]], on="trace_id", validate="many_to_one")
    data = data.reset_index(drop=True)
    group_col = "recording_date" if split_col == "date_fold" else "trace_id"
    test_values = [1] if split_col == "chrono_test" else sorted(data[split_col].unique())

    preds, selections = [], []
    for k in test_values:
        t0 = time.time()
        train = data[data[split_col] != k].reset_index(drop=True)
        test = data[data[split_col] == k].reset_index(drop=True)
        folds = _row_folds(train, group_col)

        out = test[["trace_id", "t", "application_type", "mobility_condition", "recording_date", C.TARGET]].copy()
        out["fold"] = k
        out["persistence"] = test["throughput_lag_0"].to_numpy()

        tuned = {}
        for fam, grid in GRIDS.items():
            params, score, all_scores = _inner_scores(train, folds, fam, "base", grid)
            tuned[fam] = (params, score)
            selections.append({"fold": k, "feature_set": "base", "family": fam, "params": params,
                               "inner_mae": score, "grid_scores": all_scores})
            out[f"base_{fam}"] = make_model(fam, params, "base").fit(train, train[C.TARGET]).predict(test)

        chosen = min(tuned, key=lambda f: tuned[f][1])
        out["base_selected"] = out[f"base_{chosen}"]
        params, score, all_scores = _inner_scores(train, folds, chosen, "context", GRIDS[chosen])
        selections.append({"fold": k, "feature_set": "context", "family": chosen, "params": params,
                           "inner_mae": score, "grid_scores": all_scores})
        out["context_selected"] = make_model(chosen, params, "context").fit(train, train[C.TARGET]).predict(test)
        out["selected_family"] = chosen
        preds.append(out)
        log(f"  fold {k}: selected {chosen}, {len(train)} train / {len(test)} test rows, {time.time() - t0:.0f} s")
    return pd.concat(preds, ignore_index=True), pd.DataFrame(selections)


In [ ]:
%%writefile run_pipeline.py
"""Run steps 1-4 of the synopsis pipeline plus the persistence baseline.

Usage (from the analysis folder):  python run_pipeline.py
Outputs: data/interim, data/processed, outputs/tables, outputs/run_info.json
"""
import json
import platform
from datetime import datetime

import numpy as np
import pandas as pd
import sklearn

from tput import config as C
from tput.checks import check_no_leakage, check_splits
from tput.clean import clean_all
from tput.evaluate import persistence, summarise, trace_errors
from tput.features import build_features, coverage
from tput.manifest import build_manifest


def main():
    for p in (C.INTERIM, C.PROCESSED, C.TABLES, C.FIGURES):
        p.mkdir(parents=True, exist_ok=True)

    print("1/5 manifest")
    manifest = build_manifest()
    manifest.to_csv(C.PROCESSED / "trace_manifest.csv", index=False)

    print("2/5 clean")
    clean, log = clean_all(manifest)
    clean.to_parquet(C.INTERIM / "clean_1s.parquet", index=False)
    log.to_csv(C.TABLES / "cleaning_log.csv", index=False)

    print("3/5 features")
    feats = build_features(clean)
    check_no_leakage(clean, feats)
    feats.to_parquet(C.PROCESSED / "features.parquet", index=False)
    cov = coverage(feats)
    cov.to_csv(C.TABLES / "coverage_by_group.csv", index=False)

    print("4/5 splits")
    from tput.splits import make_splits
    elig = feats[feats.eligible]
    splits = make_splits(manifest, elig.trace_id.unique())
    check_splits(splits)
    splits.to_csv(C.PROCESSED / "splits.csv", index=False)
    fold_mix = pd.crosstab([splits.application_type, splits.mobility_condition], splits.outer_fold)
    fold_mix.to_csv(C.TABLES / "outer_fold_composition.csv")

    print("5/5 persistence baseline")
    per_trace = trace_errors(elig, persistence(elig), "persistence")
    per_trace.to_csv(C.TABLES / "persistence_by_trace.csv", index=False)
    summary = summarise(per_trace, by=("mobility_condition", "application_type"))
    summary.to_csv(C.TABLES / "persistence_summary.csv", index=False)

    info = {
        "run_at": datetime.now().isoformat(timespec="seconds"),
        "input_sha256": manifest.attrs["sha256"],
        "python": platform.python_version(), "pandas": pd.__version__, "numpy": np.__version__,
        "scikit_learn": sklearn.__version__, "seed": C.SEED,
        "traces": len(manifest), "raw_rows": int(manifest.n_raw_rows.sum()),
        "seconds_after_cleaning": len(clean), "origins": len(feats), "eligible_origins": int(feats.eligible.sum()),
        "eligible_traces": int(elig.trace_id.nunique()),
    }
    (C.ROOT / "outputs" / "run_info.json").write_text(json.dumps(info, indent=2))

    pd.set_option("display.width", 160)
    print("\n", json.dumps(info, indent=2))
    print("\nCleaning totals:\n", log.drop(columns="trace_id").sum().to_string())
    print("\nCoverage:\n", cov.to_string(index=False))
    print("\nOuter folds:\n", fold_mix.to_string())
    print("\nPersistence baseline (Mbps, trace-weighted):\n", summary.to_string(index=False))


if __name__ == "__main__":
    main()


In [ ]:
%%writefile run_rq1.py
"""RQ1 tables: within-trace Spearman correlations (primary) and the active-seconds check (exploratory).

Usage (after run_pipeline.py):  python run_rq1.py
"""
import pandas as pd

from tput import config as C
from tput.rq1 import summarise, trace_correlations


def main():
    clean = pd.read_parquet(C.INTERIM / "clean_1s.parquet")
    runs = {"": clean,
            "_exploratory_active_seconds": clean[(clean.State == "D") & (clean.throughput_mbps > 0)]}
    for suffix, data in runs.items():
        corr = trace_correlations(data)
        summ = summarise(corr)
        corr.to_csv(C.TABLES / f"rq1{suffix}_trace_correlations.csv", index=False)
        summ.to_csv(C.TABLES / f"rq1{suffix}_summary.csv", index=False)
        print(f"rq1{suffix}:\n", summ[["measurement", "eligible_traces", "mean_rho", "ci95_low", "ci95_high",
                                       "p_holm"]].round(4).to_string(index=False))


if __name__ == "__main__":
    main()


In [ ]:
%%writefile run_models.py
"""Weeks 5-6: nested grouped CV for RQ2/RQ3, saving held-out predictions.

Usage (from the analysis folder, after run_pipeline.py):
  python run_models.py                    # primary: 5 outer folds of whole traces
  python run_models.py --split date_fold  # sensitivity: whole recording dates held out
  python run_models.py --split chrono_test  # sensitivity: latest dates held out
"""
import argparse
import json
import time

import pandas as pd

from tput import config as C
from tput.models import GRIDS, run_nested_cv


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--split", default="outer_fold", choices=["outer_fold", "date_fold", "chrono_test"])
    a = ap.parse_args()

    feats = pd.read_parquet(C.PROCESSED / "features.parquet")
    splits = pd.read_csv(C.PROCESSED / "splits.csv")
    out_dir = C.ROOT / "outputs" / "models"
    out_dir.mkdir(parents=True, exist_ok=True)

    t0 = time.time()
    print(f"nested CV on {a.split}")
    preds, sel = run_nested_cv(feats, splits, a.split)
    preds.to_parquet(out_dir / f"predictions_{a.split}.parquet", index=False)
    sel.assign(params=sel.params.map(json.dumps), grid_scores=sel.grid_scores.map(json.dumps)) \
        .to_csv(out_dir / f"selections_{a.split}.csv", index=False)
    (out_dir / f"run_{a.split}.json").write_text(json.dumps(
        {"split": a.split, "grids": GRIDS, "seed": C.SEED, "minutes": round((time.time() - t0) / 60, 1),
         "rows": len(preds), "traces": int(preds.trace_id.nunique())}, indent=2))
    print(f"done in {(time.time() - t0) / 60:.1f} min")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile analyze_models.py
"""RQ2-RQ4 results from saved held-out predictions (run run_models.py first).

Usage:  python analyze_models.py [--split outer_fold|date_fold|chrono_test]
Intervals are approximate: they resample saved trace errors and do not cover model-retraining
variability (synopsis 5.5). The date-cluster interval resamples whole recording dates.
"""
import argparse

import numpy as np
import pandas as pd

from tput import config as C
from tput.stats import bootstrap_diff, bootstrap_mean

MODELS = ["persistence", "base_ridge", "base_rf", "base_hgb", "base_selected", "context_selected"]
# Reference baselines added after the primary results were seen (not in the synopsis plan): video traces are
# mostly idle, so "always 0" and the 10 s mean show how much of the gain over persistence is that effect.
REFERENCE = ["ref_zero", "ref_mean_10s"]
KEYS = ["trace_id", "application_type", "mobility_condition", "recording_date"]


def per_trace(p):
    rows = []
    for m in MODELS + REFERENCE:
        e = p[KEYS].assign(ae=(p[m] - p[C.TARGET]).abs(), se=(p[m] - p[C.TARGET]) ** 2)
        g = e.groupby(KEYS)
        rows.append(pd.DataFrame({"n": g.size(), "mae": g.ae.mean(), "rmse": np.sqrt(g.se.mean())})
                    .reset_index().assign(model=m))
    return pd.concat(rows, ignore_index=True)


def paired(errs, better, worse):
    """Per-trace MAE(worse) - MAE(better); positive = 'better' has lower error."""
    w = errs.pivot_table(index=KEYS, columns="model", values="mae").reset_index()
    return w.assign(delta=w[worse] - w[better], rel=(w[worse] - w[better]) / w[worse].replace(0, np.nan))


def effect_row(name, d):
    b = bootstrap_mean(d.delta)
    bd = bootstrap_mean(d.delta, clusters=d.recording_date)
    return {"comparison": name, "traces": b["n"], "mean_delta_mbps": b["mean"], "ci95_low": b["ci_low"],
            "ci95_high": b["ci_high"], "date_ci95_low": bd["ci_low"], "date_ci95_high": bd["ci_high"],
            "boot_share_le_0": b["share_boot_le_0"], "traces_improved_pct": 100 * (d.delta > 0).mean(),
            "median_rel_improvement_pct": 100 * d.rel.median()}


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--split", default="outer_fold")
    a = ap.parse_args()
    out = C.TABLES / a.split
    out.mkdir(parents=True, exist_ok=True)

    p = pd.read_parquet(C.ROOT / "outputs" / "models" / f"predictions_{a.split}.parquet")
    hist = pd.read_parquet(C.PROCESSED / "features.parquet", columns=["trace_id", "t", "throughput_mean_10s"])
    p = p.merge(hist, on=["trace_id", "t"], how="left", validate="one_to_one")
    p["ref_zero"] = 0.0
    p["ref_mean_10s"] = p.throughput_mean_10s
    errs = per_trace(p)
    errs.to_csv(out / "trace_errors.csv", index=False)

    # Overall error table: trace-weighted (primary) and row-weighted (supplementary)
    tw = errs.groupby("model").agg(traces=("trace_id", "nunique"), mean_trace_mae=("mae", "mean"),
                                   mean_trace_rmse=("rmse", "mean"))
    rw = pd.Series({m: (p[m] - p[C.TARGET]).abs().mean() for m in MODELS + REFERENCE}, name="row_weighted_mae")
    overall = tw.join(rw).loc[MODELS + REFERENCE].round(3)
    overall.to_csv(out / "errors_overall.csv")
    by_app = errs.pivot_table(index="application_type", columns="model", values="mae")[MODELS + REFERENCE].round(3)
    by_app.to_csv(out / "errors_by_application.csv")

    # Robustness: the selected approach against the reference baselines (exploratory, added post hoc)
    ref = pd.DataFrame([effect_row(f"{r} - base_selected", paired(errs, "base_selected", r)) for r in REFERENCE]
                       + [effect_row(f"{r} - context_selected", paired(errs, "context_selected", r))
                          for r in REFERENCE])
    ref.round(3).to_csv(out / "robustness_vs_reference_baselines.csv", index=False)

    # RQ2: every family and the fold-selected approach vs persistence (base features)
    rq2 = pd.DataFrame([effect_row(f"persistence - {m}", paired(errs, m, "persistence"))
                        for m in ["base_ridge", "base_rf", "base_hgb", "base_selected"]])
    rq2.round(3).to_csv(out / "rq2_improvement_over_persistence.csv", index=False)

    # RQ3: context benefit within the selected family
    rq3 = pd.DataFrame([effect_row("base_selected - context_selected",
                                   paired(errs, "context_selected", "base_selected"))])
    rq3.round(3).to_csv(out / "rq3_context_benefit.csv", index=False)

    # RQ4: improvement of the context-augmented selected approach over persistence, by group
    d = paired(errs, "context_selected", "persistence")
    groups = []
    for col in ("mobility_condition", "application_type"):
        for g, x in d.groupby(col):
            r = effect_row(f"{col}={g}", x)
            r["mean_mae_persistence"] = x.persistence.mean()
            r["mean_mae_model"] = x.context_selected.mean()
            groups.append(r)
    pd.DataFrame(groups).round(3).to_csv(out / "rq4_by_group.csv", index=False)

    comps = []
    s, dr = d[d.mobility_condition == "Static"].delta, d[d.mobility_condition == "Driving"].delta
    comps.append({"comparison": "Driving - Static", "level": 0.95, **bootstrap_diff(dr, s)})
    apps = sorted(d.application_type.unique())
    lvl = 1 - 0.05 / 3                                                   # Bonferroni, three pairs
    for i in range(len(apps)):
        for j in range(i + 1, len(apps)):
            comps.append({"comparison": f"{apps[i]} - {apps[j]}", "level": round(lvl, 4),
                          **bootstrap_diff(d[d.application_type == apps[i]].delta,
                                           d[d.application_type == apps[j]].delta, level=lvl)})
    pd.DataFrame(comps).round(3).to_csv(out / "rq4_group_differences.csv", index=False)

    sel = pd.read_csv(C.ROOT / "outputs" / "models" / f"selections_{a.split}.csv")
    chosen = p.groupby("fold").selected_family.first()

    pd.set_option("display.width", 200)
    print(f"== {a.split} ==\nSelected family per fold: {chosen.to_dict()}")
    print("\nErrors (Mbps):\n", overall.to_string())
    print("\nRQ2 improvement over persistence (Mbps, + = AI/ML better):\n", rq2.round(3).to_string(index=False))
    print("\nErrors by application (trace-mean MAE):\n", by_app.to_string())
    print("\nRobustness vs reference baselines (post hoc):\n", ref.round(3).to_string(index=False))
    print("\nRQ3 context benefit:\n", rq3.round(3).to_string(index=False))
    print("\nRQ4 by group:\n", pd.DataFrame(groups).round(3).to_string(index=False))
    print("\nRQ4 differences:\n", pd.DataFrame(comps).round(3).to_string(index=False))
    print("\nTuned params:\n", sel[["fold", "feature_set", "family", "params", "inner_mae"]].to_string(index=False))


if __name__ == "__main__":
    main()


In [ ]:
%%writefile make_figures.py
"""Weeks 7-8: report figures (static PNG, 300 dpi) from saved results.

Usage (after run_pipeline.py, rq1 tables, run_models.py and analyze_models.py):
  python make_figures.py
Writes outputs/figures/fig*.png. Captions and notes are in outputs/figures/captions.md.
Colours: validated categorical slots 1-2 (blue, orange) on the light surface; reference baselines in neutral grey.
"""
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from tput import config as C
from tput.stats import bootstrap_mean

BLUE, ORANGE = "#2a78d6", "#eb6834"
INK, INK2, MUTED, GRID, SURFACE = "#0b0b0b", "#52514e", "#8a8985", "#e6e5e0", "#fcfcfb"
OUT = C.FIGURES
SPLIT = "outer_fold"

plt.rcParams.update({
    "font.family": "Arial", "font.size": 9, "axes.titlesize": 9, "axes.labelsize": 9,
    "axes.edgecolor": MUTED, "axes.labelcolor": INK2, "xtick.color": INK2, "ytick.color": INK2,
    "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True, "grid.color": GRID,
    "grid.linewidth": 0.6, "axes.axisbelow": True, "figure.facecolor": SURFACE, "axes.facecolor": SURFACE,
    "savefig.facecolor": SURFACE, "legend.frameon": False, "text.color": INK,
})
LABEL = {"persistence": "Persistence", "base_ridge": "Ridge", "base_rf": "Random forest",
         "base_hgb": "Gradient boosting", "context_selected": "Gradient boosting + context",
         "ref_zero": "Always 0 (reference)", "ref_mean_10s": "10 s mean (reference)"}
rng = np.random.default_rng(C.SEED)


def save(fig, name):
    fig.savefig(OUT / name, dpi=300, bbox_inches="tight")
    plt.close(fig)
    print("wrote", name)


def jitter(n, width=0.18):
    return rng.uniform(-width, width, n)


def fig1_rq1():
    corr = pd.read_csv(C.TABLES / "rq1_trace_correlations.csv")
    corr = corr[corr.eligible]
    summ = pd.read_csv(C.TABLES / "rq1_summary.csv").set_index("measurement")   # same t-intervals as the table
    fig, ax = plt.subplots(figsize=(6.2, 3.0))
    for i, m in enumerate(C.RADIO):
        c = corr[corr.measurement == m]
        for mob, col in (("Driving", BLUE), ("Static", ORANGE)):
            x = c[c.mobility_condition == mob].rho
            ax.scatter(x, i + jitter(len(x)), s=14, color=col, alpha=0.75, linewidths=0.4, edgecolors=SURFACE,
                       label=mob if i == 0 else None)
        s = summ.loc[m]
        b = {"mean": s.mean_rho, "ci_low": s.ci95_low, "ci_high": s.ci95_high}
        ax.errorbar(b["mean"], i, xerr=[[b["mean"] - b["ci_low"]], [b["ci_high"] - b["mean"]]], fmt="D",
                    color=INK, ms=5, capsize=3, lw=1.2, zorder=5, label="Mean, 95% CI" if i == 0 else None)
        ax.annotate(f"mean {b['mean']:+.2f}\n[{b['ci_low']:+.2f}, {b['ci_high']:+.2f}]", (1.01, i), va="center",
                    fontsize=7.5, color=INK2, xycoords=("axes fraction", "data"), annotation_clip=False)
    ax.axvline(0, color=MUTED, lw=0.8)
    ax.set_yticks(range(len(C.RADIO)), C.RADIO)
    ax.set_ylim(-0.6, len(C.RADIO) - 0.4)
    ax.invert_yaxis()
    ax.set_xlabel("Within-trace Spearman correlation with contemporaneous throughput")
    ax.grid(axis="y", visible=False)
    ax.legend(loc="upper left", bbox_to_anchor=(0, 1.16), ncol=3, fontsize=8, handletextpad=0.3)
    save(fig, "fig1_rq1_correlations.png")


def _paired(errs, model, ref="persistence"):
    w = errs.pivot_table(index=["trace_id", "application_type", "mobility_condition", "recording_date"],
                         columns="model", values="mae").reset_index()
    return w.assign(delta=w[ref] - w[model], rel=100 * (w[ref] - w[model]) / w[ref])


def fig2_rq2(errs):
    rows = ["base_ridge", "base_rf", "base_hgb", "context_selected", "ref_mean_10s", "ref_zero"]
    fig, ax = plt.subplots(figsize=(6.2, 2.9))
    for i, m in enumerate(rows):
        d = _paired(errs, m)
        b = bootstrap_mean(d.delta)
        ref = m.startswith("ref_")
        col = MUTED if ref else BLUE
        ax.errorbar(b["mean"], i, xerr=[[b["mean"] - b["ci_low"]], [b["ci_high"] - b["mean"]]], fmt="o",
                    color=col, ms=6, capsize=3, lw=1.4)
        ax.annotate(f"{b['mean']:+.2f} Mbps · better on {100 * (d.delta > 0).mean():.0f}% of traces",
                    (b["ci_high"], i), xytext=(6, 0), textcoords="offset points", va="center", fontsize=7.5,
                    color=INK2)
    ax.axvline(0, color=MUTED, lw=0.8)
    ax.axhline(3.5, color=GRID, lw=1)
    ax.set_yticks(range(len(rows)), [LABEL[m] for m in rows])
    for t, m in zip(ax.get_yticklabels(), rows):
        t.set_color(MUTED if m.startswith("ref_") else INK)
    ax.invert_yaxis()
    ax.set_xlim(-4.5, 6)
    ax.set_xlabel("Mean reduction in trace-level MAE vs persistence (Mbps), 95% CI  →  better")
    ax.grid(axis="y", visible=False)
    save(fig, "fig2_rq2_improvement.png")


def fig3_rq3(errs):
    d = _paired(errs, "context_selected", ref="base_selected")
    b = bootstrap_mean(d.delta)
    fig, ax = plt.subplots(figsize=(6.2, 2.6))
    lim = np.ceil(np.abs(d.delta).max() * 10) / 10
    ax.hist(d.delta, bins=np.linspace(-lim, lim, 41), color=BLUE, edgecolor=SURFACE, linewidth=1)
    ax.axvline(0, color=MUTED, lw=0.8)
    ax.axvline(b["mean"], color=INK, lw=1.2, ls="--")
    ax.annotate(f"mean {b['mean']:+.2f} Mbps (95% CI {b['ci_low']:.2f} to {b['ci_high']:.2f})\n"
                f"context better on {100 * (d.delta > 0).mean():.0f}% of {len(d)} traces",
                (b["mean"], 0.95), xycoords=("data", "axes fraction"), xytext=(8, 0), textcoords="offset points",
                va="top", fontsize=8, color=INK2)
    ax.set_xlabel("Context benefit per trace: base MAE − context MAE (Mbps)  →  context better")
    ax.set_ylabel("Traces")
    ax.grid(axis="x", visible=False)
    save(fig, "fig3_rq3_context_benefit.png")


def fig4_rq4(errs):
    d = _paired(errs, "context_selected")
    panels = [("mobility_condition", ["Static", "Driving"], "Mobility (primary)"),
              ("application_type", ["Amazon Prime", "Netflix", "Download"], "Application (exploratory)")]
    fig, axes = plt.subplots(1, 2, figsize=(6.4, 3.0), sharey=True, gridspec_kw={"width_ratios": [2, 3]})
    for ax, (col, groups, title) in zip(axes, panels):
        for i, g in enumerate(groups):
            x = d[d[col] == g]
            ax.scatter(i + jitter(len(x)), x.rel, s=14, color=BLUE, alpha=0.7, linewidths=0.4, edgecolors=SURFACE)
            med = x.rel.median()
            ax.plot([i - 0.28, i + 0.28], [med, med], color=INK, lw=1.6)
            ax.annotate(f"median {med:.0f}%\nn = {len(x)}", (i, 1.0), xycoords=("data", "axes fraction"),
                        ha="center", va="bottom", fontsize=7.5, color=INK2)
        ax.axhline(0, color=MUTED, lw=0.8)
        ax.set_xticks(range(len(groups)), groups)
        ax.set_xlim(-0.6, len(groups) - 0.4)
        ax.set_title(title, pad=26, color=INK2)
        ax.grid(axis="x", visible=False)
    axes[0].set_ylabel("MAE reduction vs persistence (%)")
    save(fig, "fig4_rq4_groups.png")


def fig5_coverage():
    f = pd.read_parquet(C.PROCESSED / "features.parquet")
    lags = [f"throughput_lag_{k}" for k in C.LAGS_S]
    strict = f[C.TARGET].notna() & f[lags].notna().all(axis=1) & (f.window_n_obs == C.WINDOW_S)
    g = f.application_type + " · " + f.mobility_condition
    cov = pd.DataFrame({"strict": 100 * strict.groupby(g).mean(), "used": 100 * f.eligible.groupby(g).mean()})
    cov = cov.sort_values("used")
    fig, ax = plt.subplots(figsize=(6.2, 2.9))
    y = np.arange(len(cov))
    h = 0.36
    ax.barh(y + h / 2 + 0.01, cov.used, h, color=BLUE, label=f"Rule used (window ≥ {C.WINDOW_MIN_OBS} of 10 s)")
    ax.barh(y - h / 2 - 0.01, cov.strict, h, color=ORANGE, label="Strict rule (all 10 s and every lag)")
    for yy, v in zip(y, cov.used):
        ax.annotate(f"{v:.0f}%", (v, yy + h / 2), xytext=(3, 0), textcoords="offset points", va="center",
                    fontsize=7.5, color=INK2)
    for yy, v in zip(y, cov.strict):
        ax.annotate(f"{v:.0f}%", (v, yy - h / 2), xytext=(3, 0), textcoords="offset points", va="center",
                    fontsize=7.5, color=INK2)
    ax.set_yticks(y, cov.index)
    ax.set_xlim(0, 100)
    ax.set_xlabel("Prediction origins kept (% of logged seconds)")
    ax.grid(axis="y", visible=False)
    ax.legend(loc="upper left", bbox_to_anchor=(0, 1.18), ncol=2, fontsize=8)
    save(fig, "fig5_coverage_rules.png")


def fig6_example():
    p = pd.read_parquet(C.ROOT / "outputs" / "models" / f"predictions_{SPLIT}.parquet")
    p = p[(p.application_type == "Download") & (p.mobility_condition == "Driving")].sort_values(["trace_id", "t"])
    # Most variable 180 s stretch of observed throughput among the held-out driving download traces
    best = None
    for tid, d in p.groupby("trace_id"):
        d = d.reset_index(drop=True)
        if len(d) < 200:
            continue
        sd = d[C.TARGET].rolling(180).std()
        i = int(sd.idxmax())
        if best is None or sd[i] > best[0]:
            best = (sd[i], d.iloc[i - 179:i + 1])
    d = best[1]
    sec = (d.t - d.t.iloc[0]).dt.total_seconds() + C.HORIZON_S          # time of the predicted value
    fig, ax = plt.subplots(figsize=(6.4, 2.8))
    series = [(C.TARGET, "Observed", INK, 1.6), ("persistence", "Persistence", ORANGE, 1.3),
              ("context_selected", "Gradient boosting + context", BLUE, 1.3)]
    for col, name, colr, lw in series:
        ax.plot(sec, d[col], color=colr, lw=lw, label=name)
    ax.set_xlabel("Time within excerpt (s)")
    ax.set_ylabel("Downlink throughput (Mbps)")
    ax.set_ylim(bottom=0)
    ax.legend(loc="upper left", bbox_to_anchor=(0, 1.16), ncol=3, fontsize=8)
    save(fig, "fig6_example_trace.png")


def fig7_sensitivity():
    """Each finding under the three evaluation schemes: does its interval stay clear of zero?"""
    schemes = [("outer_fold", "Trace folds (primary)"), ("date_fold", "Recording dates held out"),
               ("chrono_test", "Latest dates held out")]
    panels = [("rq2_improvement_over_persistence.csv", "comparison", "persistence - base_selected",
               ("mean_delta_mbps", "ci95_low", "ci95_high"), "RQ2\nImprovement over persistence"),
              ("rq3_context_benefit.csv", "comparison", "base_selected - context_selected",
               ("mean_delta_mbps", "ci95_low", "ci95_high"), "RQ3\nContext benefit"),
              ("rq4_group_differences.csv", "comparison", "Driving - Static",
               ("diff", "ci_low", "ci_high"), "RQ4a\nDriving minus static")]
    fig, axes = plt.subplots(1, 3, figsize=(6.6, 2.6), sharey=True, gridspec_kw={"wspace": 0.25})
    for ax, (fname, key, row, (m, lo, hi), title) in zip(axes, panels):
        for i, (split, _) in enumerate(schemes):
            t = pd.read_csv(C.TABLES / split / fname)
            r = t[t[key] == row].iloc[0]
            col = BLUE if split == "outer_fold" else MUTED
            ax.errorbar(r[m], i, xerr=[[r[m] - r[lo]], [r[hi] - r[m]]], fmt="o", color=col, ms=5, capsize=2.5,
                        lw=1.3)
            ax.annotate(f"{r[m]:+.2f}", (r[m], i), xytext=(0, 6), textcoords="offset points", ha="center",
                        fontsize=7, color=INK2)
        ax.axvline(0, color=MUTED, lw=0.8)
        ax.set_title(title, fontsize=8, color=INK2, loc="left")
        ax.grid(axis="y", visible=False)
        ax.margins(x=0.15)
    axes[0].set_yticks(range(len(schemes)), [s[1] for s in schemes])
    axes[0].set_ylim(-0.6, len(schemes) - 0.4)
    axes[0].invert_yaxis()
    fig.supxlabel("Mbps, 95% CI (positive = AI/ML or context better; RQ4a: driving gains more)", fontsize=8,
                  color=INK2, y=-0.04)
    save(fig, "fig7_sensitivity.png")


def main():
    OUT.mkdir(parents=True, exist_ok=True)
    errs = pd.read_csv(C.TABLES / SPLIT / "trace_errors.csv")
    fig1_rq1()
    fig2_rq2(errs)
    fig3_rq3(errs)
    fig4_rq4(errs)
    fig5_coverage()
    fig6_example()
    if all((C.TABLES / s / "rq3_context_benefit.csv").exists() for s in ("date_fold", "chrono_test")):
        fig7_sensitivity()
    else:
        print("skipped fig7_sensitivity: run run_models.py + analyze_models.py for date_fold and chrono_test")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile make_eda.py
"""Exploratory data analysis: summary tables and figures for the report (run after run_pipeline.py).

Usage:  python make_eda.py
Writes outputs/tables/eda_*.csv and outputs/figures/eda*.png.
Raw-row statistics use the unmerged CSVs (what the dataset contains); time-series statistics use the
cleaned one-row-per-second table (what the models see).
"""
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from make_figures import BLUE, GRID, INK, INK2, MUTED, ORANGE, SURFACE, save
from tput import config as C
from tput.clean import load_trace

GROUPS = [("Amazon Prime", "Static"), ("Amazon Prime", "Driving"), ("Netflix", "Static"),
          ("Netflix", "Driving"), ("Download", "Static"), ("Download", "Driving")]
MOB_COL = {"Driving": BLUE, "Static": ORANGE}


def raw_rows():
    m = pd.read_csv(C.PROCESSED / "trace_manifest.csv")
    frames = [load_trace(C.RAW_DIR / t).assign(trace_id=t) for t in m.trace_id]
    return pd.concat(frames, ignore_index=True)


def tables(raw, clean):
    # 1. Overview by application x mobility (cleaned seconds)
    g = clean.groupby(["application_type", "mobility_condition"])
    tr = clean.groupby("trace_id").agg(app=("application_type", "first"), mob=("mobility_condition", "first"),
                                       secs=("t", "size"))
    over = pd.DataFrame({
        "traces": g.trace_id.nunique(),
        "seconds": g.size(),
        "median_trace_minutes": tr.groupby(["app", "mob"]).secs.median().div(60).round(1).rename_axis(
            ["application_type", "mobility_condition"]),
        "mean_mbps": g.throughput_mbps.mean().round(2),
        "median_mbps": g.throughput_mbps.median().round(2),
        "p90_mbps": g.throughput_mbps.quantile(0.9).round(2),
        "zero_share_pct": g.throughput_mbps.apply(lambda s: round(100 * (s == 0).mean(), 1)),
        "share_5g_pct": g.NetworkMode.apply(lambda s: round(100 * (s == "5G").mean(), 1)),
        "mean_speed_kmh": g.Speed.mean().round(1),
    }).reindex(pd.MultiIndex.from_tuples(GROUPS, names=["application_type", "mobility_condition"]))
    over.to_csv(C.TABLES / "eda_overview_by_group.csv")

    # 2. Descriptive statistics of the modelling variables (cleaned seconds)
    cols = {"throughput_mbps": "Throughput (Mbps)", "RSRP": "RSRP (dBm)", "RSRQ": "RSRQ (dB)", "SNR": "SNR (dB)",
            "CQI": "CQI", "Speed": "Speed (km/h)"}
    desc = clean[list(cols)].describe(percentiles=[0.25, 0.5, 0.75]).T
    desc = desc.assign(missing_pct=(100 * clean[list(cols)].isna().mean()).round(1))
    desc.index = [cols[c] for c in desc.index]
    desc = desc[["count", "missing_pct", "mean", "std", "min", "25%", "50%", "75%", "max"]].round(2)
    desc.to_csv(C.TABLES / "eda_descriptive_stats.csv")

    # 3. Missingness and data-quality issues in the raw rows
    num = ["RSRP", "RSRQ", "SNR", "CQI", "RSSI", "NRxRSRP", "NRxRSRQ", "PINGAVG", "Speed", "DL_bitrate"]
    miss = (100 * raw[num].isna().mean()).round(1).rename("missing_pct_raw_rows")
    miss.to_csv(C.TABLES / "eda_missingness.csv")

    steps = clean.groupby("trace_id").t.diff().dt.total_seconds().dropna()
    quality = pd.Series({
        "raw_rows": len(raw),
        "rows_with_repeated_timestamp": int(raw.duplicated(["trace_id", "t"], keep=False).sum()),
        "rsrp_sentinel_minus200": int((raw.RSRP == -200).sum()),
        "neighbour_rsrp_sentinel_minus205": int((raw.NRxRSRP == -205).sum()),
        "snr_below_minus23": int((raw.SNR < -23).sum()),
        "cqi_zero": int((raw.CQI == 0).sum()),
        "seconds_after_merge": len(clean),
        "gaps_2s": int((steps == 2).sum()),
        "gaps_3s_plus": int((steps >= 3).sum()),
        "max_dl_mbps": round(raw.DL_bitrate.max() / 1000, 1),
        "max_speed_kmh": raw.Speed.max(),
    })
    quality.to_csv(C.TABLES / "eda_data_quality.csv", header=["value"])

    # 4. Network-mode composition by mobility (cleaned seconds)
    mode = clean.NetworkMode.where(clean.NetworkMode.isin(["5G", "LTE"]), "3G/2G")
    comp = (pd.crosstab(clean.mobility_condition, mode, normalize="index") * 100).round(1)[["5G", "LTE", "3G/2G"]]
    comp.to_csv(C.TABLES / "eda_network_mode_by_mobility.csv")
    return over, desc, miss, quality, comp


def fig_throughput(clean):
    fig, axes = plt.subplots(1, 2, figsize=(6.6, 2.9), gridspec_kw={"width_ratios": [3, 2]})
    ax = axes[0]
    for i, (app, mob) in enumerate(GROUPS):
        s = clean[(clean.application_type == app) & (clean.mobility_condition == mob)].throughput_mbps
        nz = s[s > 0]
        ax.boxplot(nz, positions=[i], widths=0.55, showfliers=False, patch_artist=True,
                   boxprops=dict(facecolor=MOB_COL[mob], alpha=0.75, edgecolor=MOB_COL[mob]),
                   medianprops=dict(color=INK, lw=1.4), whiskerprops=dict(color=MUTED), capprops=dict(color=MUTED))
    ax.set_yscale("log")
    ax.set_xticks(range(len(GROUPS)), [f"{a.replace('Amazon ', '')}\n{m}" for a, m in GROUPS], fontsize=7.5)
    ax.set_ylabel("Non-zero throughput (Mbps, log scale)")
    ax.set_title("Throughput when data is flowing", fontsize=8.5, color=INK2, loc="left")
    ax.grid(axis="x", visible=False)
    ax = axes[1]
    zs = [100 * (clean[(clean.application_type == a) & (clean.mobility_condition == m)].throughput_mbps == 0).mean()
          for a, m in GROUPS]
    y = np.arange(len(GROUPS))
    ax.barh(y, zs, color=[MOB_COL[m] for _, m in GROUPS], height=0.6)
    for yy, v in zip(y, zs):
        ax.annotate(f"{v:.0f}%", (v, yy), xytext=(3, 0), textcoords="offset points", va="center", fontsize=7.5,
                    color=INK2)
    ax.set_yticks(y, [f"{a} · {m}" for a, m in GROUPS], fontsize=7.5)
    ax.invert_yaxis()
    ax.set_xlim(0, 100)
    ax.set_xlabel("Seconds with zero throughput (%)")
    ax.set_title("Idle seconds", fontsize=8.5, color=INK2, loc="left")
    ax.grid(axis="y", visible=False)
    fig.tight_layout()
    save(fig, "eda1_throughput_by_group.png")


def fig_radio(clean):
    cols = [("RSRP", "RSRP (dBm)"), ("RSRQ", "RSRQ (dB)"), ("SNR", "SNR (dB)"), ("CQI", "CQI")]
    fig, axes = plt.subplots(1, 4, figsize=(6.8, 2.2))
    for ax, (c, lab) in zip(axes, cols):
        lo, hi = clean[c].quantile([0.005, 0.995])
        bins = np.arange(-0.5, 16, 1) if c == "CQI" else np.linspace(lo, hi, 30)
        for mob in ("Static", "Driving"):
            ax.hist(clean.loc[clean.mobility_condition == mob, c].dropna(), bins=bins, density=True,
                    histtype="step", lw=1.4, color=MOB_COL[mob], label=mob)
        ax.set_xlabel(lab, fontsize=8)
        ax.set_yticks([])
        ax.grid(False)
    axes[0].legend(loc="upper left", bbox_to_anchor=(0, 1.32), ncol=2, fontsize=7.5)
    fig.tight_layout()
    save(fig, "eda2_radio_distributions.png")


def fig_mode_and_gaps(clean, comp):
    fig, axes = plt.subplots(1, 2, figsize=(6.6, 2.3), gridspec_kw={"width_ratios": [3, 2]})
    ax = axes[0]
    shades = {"5G": BLUE, "LTE": "#86b6ef", "3G/2G": "#cde2fb"}
    for i, mob in enumerate(["Static", "Driving"]):
        left = 0
        for mode in ["5G", "LTE", "3G/2G"]:
            v = comp.loc[mob, mode]
            ax.barh(i, v, left=left, color=shades[mode], edgecolor=SURFACE, lw=1.5, height=0.55,
                    label=mode if i == 0 else None)
            if v >= 6:
                ax.text(left + v / 2, i, f"{v:.0f}%", ha="center", va="center", fontsize=7.5,
                        color="white" if mode == "5G" else INK)
            left += v
    ax.set_yticks([0, 1], ["Static", "Driving"])
    ax.set_xlim(0, 100)
    ax.set_xlabel("Share of seconds by network mode (%)")
    ax.legend(loc="upper left", bbox_to_anchor=(0, 1.3), ncol=3, fontsize=7.5)
    ax.grid(axis="y", visible=False)
    ax = axes[1]
    steps = clean.groupby("trace_id").t.diff().dt.total_seconds().dropna()
    cats = ["1 s", "2 s", "3 s+"]
    vals = [100 * (steps == 1).mean(), 100 * (steps == 2).mean(), 100 * (steps >= 3).mean()]
    ax.bar(cats, vals, color=[BLUE, ORANGE, ORANGE], width=0.6)
    for x, v in zip(cats, vals):
        ax.annotate(f"{v:.1f}%", (x, v), xytext=(0, 2), textcoords="offset points", ha="center", fontsize=7.5,
                    color=INK2)
    ax.set_ylabel("Share of consecutive records (%)")
    ax.set_xlabel("Time between records")
    ax.set_ylim(0, 100)
    ax.grid(axis="x", visible=False)
    fig.tight_layout()
    save(fig, "eda3_network_mode_and_gaps.png")


def fig_example_traces(clean):
    picks = [("Netflix", "Static"), ("Download", "Driving")]
    fig, axes = plt.subplots(2, 2, figsize=(6.8, 3.4), sharex="col", gridspec_kw={"height_ratios": [2, 1]})
    for j, (app, mob) in enumerate(picks):
        sub = clean[(clean.application_type == app) & (clean.mobility_condition == mob)]
        tid = sub.groupby("trace_id").size().sort_values().index[len(sub.trace_id.unique()) // 2]   # median length
        d = sub[sub.trace_id == tid].sort_values("t").head(300)
        sec = (d.t - d.t.iloc[0]).dt.total_seconds()
        axes[0, j].plot(sec, d.throughput_mbps, color=MOB_COL[mob], lw=1.1)
        axes[0, j].set_title(f"{app}, {mob.lower()} (300 records of one trace)", fontsize=8, color=INK2, loc="left")
        axes[1, j].plot(sec, d.CQI, color=INK2, lw=1.0, drawstyle="steps-post")
        axes[1, j].set_ylim(0, 15.5)
        axes[1, j].set_xlabel("Time (s)", fontsize=8)
    axes[0, 0].set_ylabel("Throughput (Mbps)", fontsize=8)
    axes[1, 0].set_ylabel("CQI", fontsize=8)
    fig.tight_layout()
    save(fig, "eda4_example_traces.png")


def main():
    clean = pd.read_parquet(C.INTERIM / "clean_1s.parquet")
    raw = raw_rows()
    over, desc, miss, quality, comp = tables(raw, clean)
    fig_throughput(clean)
    fig_radio(clean)
    fig_mode_and_gaps(clean, comp)
    fig_example_traces(clean)
    pd.set_option("display.width", 200)
    print(over.to_string(), "\n\n", desc.to_string(), "\n\n", miss.to_string(), "\n\n", quality.to_string(),
          "\n\n", comp.to_string())


if __name__ == "__main__":
    main()


In [ ]:
%%writefile run_importance.py
"""Permutation feature importance of the selected context-augmented model, on held-out traces.

For each outer fold the model is refitted with the fold's tuned settings (selections_outer_fold.csv) and the
increase in MAE when each input is shuffled is measured on that fold's test traces. Results are averaged
over the five folds. Usage (after run_models.py):  python run_importance.py
"""
import json

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.inspection import permutation_importance

from make_figures import BLUE, INK2, MUTED, save
from tput import config as C
from tput.models import FEATURE_SETS, make_model

LABELS = {"throughput_lag_0": "Throughput now", "throughput_lag_1": "Throughput 1 s ago",
          "throughput_lag_2": "Throughput 2 s ago", "throughput_lag_5": "Throughput 5 s ago",
          "throughput_mean_10s": "10 s mean throughput", "throughput_sd_10s": "10 s throughput SD",
          "Speed": "Speed", "NetworkMode": "Network mode", "application_type": "Application type"}
MAX_TEST_ROWS = 15000


def main():
    feats = pd.read_parquet(C.PROCESSED / "features.parquet")
    splits = pd.read_csv(C.PROCESSED / "splits.csv")
    sel = pd.read_csv(C.ROOT / "outputs" / "models" / "selections_outer_fold.csv")
    data = feats[feats.eligible].merge(splits[["trace_id", "outer_fold"]], on="trace_id")
    cols = FEATURE_SETS["context"]["num"] + FEATURE_SETS["context"]["cat"]
    rng = np.random.default_rng(C.SEED)

    rows = []
    for k in sorted(data.outer_fold.unique()):
        s = sel[(sel.fold == k) & (sel.feature_set == "context")].iloc[0]
        train, test = data[data.outer_fold != k], data[data.outer_fold == k]
        if len(test) > MAX_TEST_ROWS:
            test = test.iloc[rng.choice(len(test), MAX_TEST_ROWS, replace=False)]
        model = make_model(s.family, json.loads(s.params), "context").fit(train, train[C.TARGET])
        r = permutation_importance(model, test[cols], test[C.TARGET], scoring="neg_mean_absolute_error",
                                   n_repeats=5, random_state=C.SEED, n_jobs=-1)
        rows.append(pd.DataFrame({"feature": cols, "fold": k, "mae_increase": r.importances_mean}))
        print(f"fold {k}: {s.family} done")

    imp = pd.concat(rows)
    imp["feature"] = imp.feature.where(~imp.feature.str.endswith("_missing"), "Missing-value flags")
    summ = (imp.groupby(["feature", "fold"]).mae_increase.sum().groupby("feature")
            .agg(mean_mae_increase="mean", sd="std").sort_values("mean_mae_increase", ascending=False))
    summ.round(4).to_csv(C.TABLES / "feature_importance.csv")
    print(summ.round(4).to_string())

    s = summ.iloc[::-1]
    fig, ax = plt.subplots(figsize=(6.2, 3.6))
    y = np.arange(len(s))
    ax.barh(y, s.mean_mae_increase, xerr=s.sd, color=BLUE, height=0.6, error_kw=dict(ecolor=MUTED, lw=0.8))
    ax.set_yticks(y, [LABELS.get(f, f) for f in s.index], fontsize=8)
    ax.axvline(0, color=MUTED, lw=0.8)
    ax.set_xlabel("Increase in MAE when the input is shuffled (Mbps); bars = SD across folds")
    ax.grid(axis="y", visible=False)
    save(fig, "fig8_feature_importance.png")


if __name__ == "__main__":
    main()


## 2. Data acquisition
Downloads `5G-production-dataset.zip` from https://github.com/uccmisl/5Gdataset, verifies its SHA-256 checksum against the version used in the report, and extracts only the 83 CSV traces (the archive's `__MACOSX` files are skipped).

In [ ]:
import hashlib, urllib.request, zipfile
from tput import config as C
C.RAW_ZIP.parent.mkdir(parents=True, exist_ok=True)
if not C.RAW_ZIP.exists():
    urllib.request.urlretrieve('https://raw.githubusercontent.com/uccmisl/5Gdataset/master/5G-production-dataset.zip', C.RAW_ZIP)
digest = hashlib.sha256(C.RAW_ZIP.read_bytes()).hexdigest()
print('SHA-256 matches the report version:', digest == C.RAW_ZIP_SHA256)
with zipfile.ZipFile(C.RAW_ZIP) as z:
    members = [n for n in z.namelist() if n.startswith('5G-production-dataset/') and n.endswith('.csv')]
    z.extractall(C.RAW_ZIP.parent, members)
print(len(members), 'CSV traces extracted')

## 3. Data preparation
Builds the trace manifest, cleans the data (missing-value markers, the RSRP -200 sentinel, repeated timestamps), constructs the 5-second target and history features, assigns whole traces to evaluation folds, checks for leakage, and scores the persistence baseline.

In [ ]:
!python run_pipeline.py

## 4. Exploratory data analysis

In [ ]:
!python -W ignore make_eda.py > /dev/null
from IPython.display import Image, display
T, F = C.TABLES, C.FIGURES
display(pd.read_csv(T / 'eda_overview_by_group.csv'))
display(pd.read_csv(T / 'eda_descriptive_stats.csv', index_col=0))
display(pd.read_csv(T / 'eda_data_quality.csv', index_col=0))
for f in ['eda1_throughput_by_group.png', 'eda3_network_mode_and_gaps.png', 'eda2_radio_distributions.png', 'eda4_example_traces.png']:
    display(Image(filename=str(F / f), width=820))

**Insights.** Video throughput is governed by the player's on/off pattern (Netflix is idle in about three quarters of seconds), driving traces spend much of their time outside 5G, and about 10% of seconds are missing from the logs. These shape RQ1 (weak radio-throughput association expected) and RQ2 (history should dominate).

## 5. RQ1: radio measurements and throughput
Within-trace Spearman correlations, one-sample t tests across traces, Holm adjustment.

In [ ]:
!python run_rq1.py > /dev/null
display(pd.read_csv(T / 'rq1_summary.csv')[['measurement', 'eligible_traces', 'mean_rho', 'ci95_low', 'ci95_high', 'p_holm', 'date_p_holm']].round(4))

## 6. Models: nested grouped cross-validation (RQ2-RQ4)
Persistence vs ridge regression, random forest and gradient boosting. Five outer folds hold out whole traces; three inner folds choose the settings; the context model adds speed, network mode and application type.

In [ ]:
!python -W ignore run_models.py --split outer_fold
!python -W ignore analyze_models.py --split outer_fold > /dev/null
if RUN_SENSITIVITY:
    for s in ['date_fold', 'chrono_test']:
        !python -W ignore run_models.py --split {s}
        !python -W ignore analyze_models.py --split {s} > /dev/null

### RQ2: AI/ML compared with persistence

In [ ]:
O = T / 'outer_fold'
display(pd.read_csv(O / 'errors_overall.csv'))
display(pd.read_csv(O / 'rq2_improvement_over_persistence.csv')[['comparison', 'mean_delta_mbps', 'ci95_low', 'ci95_high', 'traces_improved_pct']])
display(pd.read_csv(O / 'robustness_vs_reference_baselines.csv')[['comparison', 'mean_delta_mbps', 'ci95_low', 'ci95_high', 'traces_improved_pct']])

### RQ3: added value of context, and RQ4: mobility and application

In [ ]:
display(pd.read_csv(O / 'rq3_context_benefit.csv')[['comparison', 'mean_delta_mbps', 'ci95_low', 'ci95_high', 'traces_improved_pct']])
display(pd.read_csv(O / 'rq4_by_group.csv')[['comparison', 'traces', 'mean_mae_persistence', 'mean_mae_model', 'mean_delta_mbps', 'ci95_low', 'ci95_high', 'median_rel_improvement_pct']])
display(pd.read_csv(O / 'rq4_group_differences.csv'))

## 7. Figures

In [ ]:
!python -W ignore make_figures.py > /dev/null
for f in ['fig1_rq1_correlations.png', 'fig2_rq2_improvement.png', 'fig6_example_trace.png', 'fig3_rq3_context_benefit.png', 'fig4_rq4_groups.png', 'fig7_sensitivity.png']:
    p = F / f
    if p.exists(): display(Image(filename=str(p), width=820))

## 8. Feature importance
Permutation importance of the selected context model on held-out traces, averaged over the five folds.

In [ ]:
if RUN_IMPORTANCE:
    !python -W ignore run_importance.py > /dev/null
    display(pd.read_csv(T / 'feature_importance.csv'))
    display(Image(filename=str(F / 'fig8_feature_importance.png'), width=760))

## 9. Summary
| RQ | Result |
|---|---|
| RQ1 | No radio measurement is associated with throughput within traces after Holm adjustment. |
| RQ2 | Gradient boosting with an absolute-error loss beats persistence on held-out traces, and also beats a 10 s moving average. |
| RQ3 | Context adds a small benefit that does not hold when whole recording days are held out. |
| RQ4 | No clear difference between stationary and driving traces; application differences follow throughput scale. |

Throughput history dominates feature importance; the four radio measurements contribute almost nothing. See the interim report for interpretation and limitations.